# ETL Driblab Capstone

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')

print('Setup complete ✓')

In [ ]:
rawevents = json.load(open("/Users/nataliaurrea/Documents/IE/MBDS/Term III/Driblab/data/raw/683231_events.json"))

df = pd.json_normalize(rawevents)

In [ ]:
df.rename(columns={
    'event.id':              'event_id',
    'event.event_type_id':   'event_type_id',
    'event.event_type_name': 'event_type_name',
    'team.team_id':          'team_id',
    'team.team_name':        'team_name',
    'player.player_id':      'player_id',
    'player.player_name':    'player_name',
}, inplace=True)
df.head()

In [ ]:
dim = pd.read_csv("/Users/nataliaurrea/Documents/IE/MBDS/Term III/Driblab/data/raw/dim_event_type.csv")
dim.head()
dim.tail()


In [ ]:
dim.rename(columns={'event_id': 'event_type_id'}, inplace=True)

In [ ]:
df = df.merge(
    dim[[ 'event_type_id', 'category_id', 'name_eng', 'des_eng']],
    on='event_type_id',
    how='left'
)
df.head()

In [ ]:
print(df.shape)

In [ ]:
print(df.columns)

In [ ]:
df.info()

In [ ]:
df.isnull().sum() 

In [ ]:
df.drop(columns=['qualifiers']).duplicated().sum() #drop qualifiers as it has a dictionary

In [ ]:
df['event_type_name'].value_counts()

In [ ]:
team_events = df.groupby(['team_name','event_type_name']).size().unstack(fill_value=0)
team_events.head()

FC Barcelona generated more passes, leading to mayority ball possesion

In [ ]:
team_events_pct = team_events.div(team_events.sum(axis=1), axis=0).mul(100).round(1)
team_events_pct.head()

In [ ]:
top_10_events = df['event_type_name'].value_counts().head(10).index
team_events_pct[top_10_events]

In [ ]:
df.groupby('event_type_name')['outcome'].mean().mul(100).round(1).sort_values(ascending=False)